In [ ]:
#### notebook for KEGG enrichment of TR-proteins in 99% dereplicated Methanosarcina proteomes

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from statsmodels.stats.multitest import multipletests

In [4]:
def extract_ids_from_fasta(fasta_path):
    """Extracts protein IDs from FASTA headers: >protein_ID description"""
    protein_ids = set()
    try:
        with open(fasta_path, 'r') as f:
            for line in f:
                if line.startswith(">"):
                    prot_id = line[1:].strip().split(" ")[0]
                    protein_ids.add(prot_id)
    except FileNotFoundError:
        print(f"Error: FASTA file not found at {fasta_path}")
    return protein_ids

def load_fasta_as_dict(fasta_path):
    """Loads FASTA into a dictionary {ID: sequence}"""
    seqs = {}
    current_id = None
    current_seq = []
    try:
        with open(fasta_path, 'r') as f:
            for line in f:
                if line.startswith(">"):
                    if current_id:
                        seqs[current_id] = "".join(current_seq)
                    current_id = line[1:].strip().split(" ")[0]
                    current_seq = []
                else:
                    current_seq.append(line.strip())
        if current_id:
            seqs[current_id] = "".join(current_seq)
    except FileNotFoundError:
        print(f"Error: FASTA file not found at {fasta_path}")
    return seqs

def get_min_circular_shift(s):
    """Normalizes circular sequences to their lexicographical minimum: HKHKA -> AHKHK"""
    if not s: return s
    # Generate all possible rotations and return the alphabetically first one
    return min(s[i:] + s[:i] for i in range(len(s)))

def parse_tr_bed(bed_path, tr_fasta_dict):
    """Parses BED file and extracts normalized unit sequences."""
    tr_data = []
    try:
        with open(bed_path, 'r') as f:
            for line in f:
                cols = line.strip().split('\t')
                if len(cols) < 11: continue
                prot_id, start = cols[0], int(cols[1])
                try:
                    unit_size = int(cols[10].split(',')[0])
                except (ValueError, IndexError): continue
                
                if prot_id in tr_fasta_dict:
                    raw_unit = tr_fasta_dict[prot_id][start : start + unit_size]
                    norm_unit = get_min_circular_shift(raw_unit)
                    tr_data.append({'protein_id': prot_id, 'unit_seq': norm_unit})
    except FileNotFoundError:
        print(f"Error: BED file not found at {bed_path}")
    return pd.DataFrame(tr_data)

def run_comprehensive_permutation_v2(bg_fasta, tr_fasta, tr_bed, protein_to_ko_file, kegg_db_file, n_perms=2000):
    # 1. Load Data
    bg_ids = extract_ids_from_fasta(bg_fasta)
    tr_fasta_dict = load_fasta_as_dict(tr_fasta)
    tr_ids = set(tr_fasta_dict.keys())
    
    # 2. Extract motifs
    tr_metadata = parse_tr_bed(tr_bed, tr_fasta_dict)
    
    # 3. Load Annotations
    prot_map = pd.read_csv(protein_to_ko_file, sep='\t') 
    kegg_db = pd.read_csv(kegg_db_file) 
    full_ann = pd.merge(prot_map[['protein_id', 'KO']], kegg_db, on='KO', how='inner')
    full_ann = full_ann.loc[:, ~full_ann.columns.duplicated()].copy()

    # --- PRINT SUMMARY BLOCK START ---
    print(f"Total proteins in Background FASTA: {len(bg_ids)}")
    print(f"Total proteins in Treatment FASTA:  {len(tr_ids)}")
    print("Loading annotation tables...")
    
    # Calculate coverage based on the merged annotation table
    bg_with_ko = full_ann[full_ann['protein_id'].isin(bg_ids)]['protein_id'].nunique()
    tr_with_ko = full_ann[full_ann['protein_id'].isin(tr_ids)]['protein_id'].nunique()
    
    print(f"Coverage: BG {bg_with_ko}/{len(bg_ids)} | TR {tr_with_ko}/{len(tr_ids)}")
    print("-" * 40)
    # --- PRINT SUMMARY BLOCK END ---
    
    levels_to_test = {'KO': 'KO', 'Level_C_Name': 'Level_C_ID', 'Level_B_Name': 'Level_B_ID'}
    
    for name_col, id_col in levels_to_test.items():
        # The rest of your loop follows...
        print(f"\n--- Analyzing Level: {name_col} ---")
        
        bg_data = full_ann[full_ann['protein_id'].isin(bg_ids)].dropna(subset=[name_col])
        tr_data_raw = full_ann[full_ann['protein_id'].isin(tr_ids)].dropna(subset=[name_col])
        
        # Merge with TR motifs and drop duplicates to match Version 1 protein counting
        tr_data = pd.merge(tr_data_raw, tr_metadata, on='protein_id', how='inner').drop_duplicates(subset=['protein_id'])
        
        n_tr = len(tr_data_raw) 
        if n_tr == 0: continue

        counts_all_bg = bg_data.groupby(name_col).size().to_dict()

        def format_tr_counts(series):
            units = [u for u in series if pd.notna(u)]
            if not units: return ""
            counts = Counter(units)
            return ", ".join([f"[{seq}]{count}" for seq, count in counts.most_common()])

        group_keys = list(set([id_col, name_col]))
        obs_df = tr_data.groupby(group_keys).agg({
            'protein_id': [('Observed_with_TRs', 'size'), 
                           ('Protein_IDs', lambda x: "; ".join(x.tolist()))],
            'unit_seq': [('aaTR_versions_and_protein_counts', format_tr_counts)],
            'KO': [('Unique_KOs', 'nunique'),
                   ('KO_Terms', lambda x: "; ".join(sorted(x.unique())))]
        })
        obs_df.columns = obs_df.columns.get_level_values(1)
        obs_df = obs_df.reset_index()
        obs_df['Total_Copies_in_Data'] = obs_df[name_col].map(counts_all_bg)

        # 5. Permutation Logic
        bg_pool = bg_data[name_col].values
        categories = obs_df[name_col].tolist()
        null_dist = {cat: np.zeros(n_perms) for cat in categories}
        
        print(f"Running {n_perms} permutations...")
        for i in range(n_perms):
            sample = np.random.choice(bg_pool, size=n_tr, replace=False)
            val, counts = np.unique(sample, return_counts=True)
            d = dict(zip(val, counts))
            for cat in categories:
                null_dist[cat][i] = d.get(cat, 0)
        
        # 6. Build Results and Sanity Check
        stats_results = []
        for _, row in obs_df.iterrows():
            cat_name = row[name_col]
            obs = row['Observed_with_TRs']
            
            # Sanity Check
            motif_string = row['aaTR_versions_and_protein_counts']
            motif_counts = sum([int(n) for n in re.findall(r'\](\d+)', motif_string)])
            id_count = len(row['Protein_IDs'].split('; '))
            check = (obs == motif_counts == id_count)
            
            nulls = null_dist[cat_name]
            m, s = np.mean(nulls), np.std(nulls)
            z = (obs - m) / s if s > 0 else 0
            p = min(1.0, 2 * min((np.sum(nulls >= obs)+1)/(n_perms+1), (np.sum(nulls <= obs)+1)/(n_perms+1)))
            
            stats_results.append({
                'ID': row[id_col],
                'Category': cat_name,
                'Observed_with_TRs': obs,
                'Total_Copies_in_Data': row['Total_Copies_in_Data'],
                'Unique_KOs': row['Unique_KOs'],
                'KO_Terms': row['KO_Terms'],
                'Z_score': round(z, 4),
                'p_value': p,
                'aaTR_versions_and_protein_counts': motif_string,
                'Protein_IDs': row['Protein_IDs'],
                'Sanity_Check_Passed': check
            })
            
        res_df = pd.DataFrame(stats_results)
        if res_df.empty: continue

        # FDR Correction
        _, p_adj, _, _ = multipletests(res_df['p_value'], method='fdr_bh')
        res_df['FDR_adj_p'] = p_adj

        # --- 7. Status Column Logic (RESTORED) ---
        conditions = [
            (res_df['FDR_adj_p'] < 0.05) & (res_df['Z_score'] > 0),
            (res_df['FDR_adj_p'] < 0.05) & (res_df['Z_score'] < 0)
        ]
        res_df['Status'] = np.select(conditions, ["Enriched", "Depleted"], default="NS")
        
        # 8. Sort by Z-score and Save
        res_df = res_df.sort_values('Z_score', ascending=False)
        out_name = f"protein_analysis_{name_col.replace('_Name','')}_v2.0.tsv"
        res_df.to_csv(out_name, sep='\t', index=False)
        
        if not res_df['Sanity_Check_Passed'].all():
            print(f"⚠️ Warning: Some rows in {name_col} failed the sanity check!")
        else:
            print(f"✅ {name_col} analysis complete. Status assigned and files saved.")

In [5]:
# --- Execution Block ---
run_comprehensive_permutation_v2(
    bg_fasta="/cluster/project/umbiol/projects/methanogens/methanosarcina/proteins/dRep_99/2026-04-14_all-28-Methanosarcina-proteomes.faa",
    tr_fasta="/cluster/project/umbiol/projects/methanogens/methanosarcina/proteins/dRep_99/TR_out_dRep-Methanosarcina-proteomes/sequences/TR_out_dRep-Methanosarcina-proteomes.sequences-with-repeats.fa",
    protein_to_ko_file="/cluster/project/umbiol/projects/methanogens/methanosarcina/proteins/protein_clustering/all_proteomes/kofam_scan/2026-04-29_kofam-scan_annotations_top-hits_filtered.tsv", # Columns: protein_id, KO 
    tr_bed="/cluster/project/umbiol/projects/methanogens/methanosarcina/proteins/dRep_99/TR_out_dRep-Methanosarcina-proteomes/annotations/TR_out_dRep-Methanosarcina-proteomes.bed",
    kegg_db_file="2026-03-10_KEGG-db_ko00001.csv", 
    n_perms=5000
)

Total proteins in Background FASTA: 95121
Total proteins in Treatment FASTA:  960
Loading annotation tables...
Coverage: BG 67661/95121 | TR 504/960
----------------------------------------

--- Analyzing Level: KO ---
Running 5000 permutations...
✅ KO analysis complete. Status assigned and files saved.

--- Analyzing Level: Level_C_Name ---
Running 5000 permutations...
✅ Level_C_Name analysis complete. Status assigned and files saved.

--- Analyzing Level: Level_B_Name ---
Running 5000 permutations...
✅ Level_B_Name analysis complete. Status assigned and files saved.


In [ ]:
import pandas as pd
import numpy as np


def analyze_zero_hit_categories(bg_fasta, tr_fasta, protein_to_ko_file, kegg_db_file):
    """
    Identifies functional groups present in the genome (background) 
    that are completely absent in the tandem repeat (treatment) set.
    """
    # 1. Load IDs
    bg_ids = extract_ids_from_fasta(bg_fasta)
    tr_ids = extract_ids_from_fasta(tr_fasta)

    # 2. Load and Prepare Annotations
    print("Loading and merging annotation tables...")
    prot_map = pd.read_csv(protein_to_ko_file, sep='\t') 
    kegg_db = pd.read_csv(kegg_db_file) 
    
    # Create the "Golden Universe" (Only proteins present in our BG FASTA)
    full_ann = pd.merge(prot_map[['protein_id', 'KO']], kegg_db, on='KO', how='inner')
    bg_ann = full_ann[full_ann['protein_id'].isin(bg_ids)].copy()
    tr_ann = full_ann[full_ann['protein_id'].isin(tr_ids)].copy()

    # 3. Global Stats for Metadata
    total_bg_genes = len(bg_ids)
    bg_with_ko = bg_ann['protein_id'].nunique()
    total_tr_genes = len(tr_ids)
    tr_with_ko = tr_ann['protein_id'].nunique()

    print(f"--- Global Coverage ---")
    print(f"Background: {bg_with_ko}/{total_bg_genes} annotated")
    print(f"Treatment:  {tr_with_ko}/{total_tr_genes} annotated")

    # 4. Define Analysis Levels
    # We use lists to ensure we keep hierarchy context (e.g., keeping B and C with KOs)
    levels_to_test = {
        'KO': ['KO', 'Level_C_Name', 'Level_B_Name'],
        'Level_C_Name': ['Level_C_ID', 'Level_C_Name', 'Level_B_Name'],
        'Level_B_Name': ['Level_B_ID', 'Level_B_Name']
    }

    for main_level, context_cols in levels_to_test.items():
        print(f"\nProcessing Zero-Hits for: {main_level}")

        # Get unique categories in Background and Treatment
        bg_cats = set(bg_ann[main_level].unique())
        tr_cats = set(tr_ann[main_level].unique())
        
        # Identify categories in BG that are NOT in TR
        missing_cats = bg_cats - tr_cats

        if not missing_cats:
            print(f"No categories at {main_level} were completely missing.")
            continue

        # Calculate counts per category in Background
        counts = bg_ann.groupby(main_level).size().reset_index(name='Total_Copies_in_Genome')

        # Create a mapping of the category to its higher hierarchy
        hierarchy_map = bg_ann[context_cols].drop_duplicates()

        # Build the final table for this level
        zero_df = hierarchy_map[hierarchy_map[main_level].isin(missing_cats)].copy()
        zero_df = pd.merge(zero_df, counts, on=main_level, how='left')

        # Add enrichment-style metrics for comparison
        zero_df['Observed_with_TRs'] = 0
        zero_df['Pct_of_Annotated_Genome'] = (zero_df['Total_Copies_in_Genome'] / bg_with_ko * 100).round(4)
        zero_df['Status'] = "Zero_Hits"

        # Sort by impact (largest groups in genome that are missing in TRs)
        zero_df = zero_df.sort_values('Total_Copies_in_Genome', ascending=False)

        # 5. Save Output
        out_name = f"zero_hit_analysis_{main_level.replace('_Name','')}.tsv"
        with open(out_name, 'w') as f:
            f.write(f"# Analysis: Zero-Hit Functional Categories\n")
            f.write(f"# Total_Genes_in_Genome: {total_bg_genes}\n")
            f.write(f"# Annotated_Genes_in_Genome: {bg_with_ko}\n")
            f.write(f"# Total_TR_Proteins: {total_tr_genes}\n")
            f.write(f"# Annotated_TR_Proteins: {tr_with_ko}\n")
            zero_df.to_csv(f, sep='\t', index=False)
        
        print(f"Successfully saved {len(zero_df)} missing categories to {out_name}")

# --- Run the analysis ---
analyze_zero_hit_categories(
    bg_fasta="/cluster/project/umbiol/projects/methanogens/methanosarcina/proteins/dRep_99/2026-04-14_all-28-Methanosarcina-proteomes.faa",
    tr_fasta="/cluster/project/umbiol/projects/methanogens/methanosarcina/proteins/dRep_99/TR_out_dRep-Methanosarcina-proteomes/sequences/TR_out_dRep-Methanosarcina-proteomes.sequences-with-repeats.fa",
    protein_to_ko_file="/cluster/project/umbiol/projects/methanogens/methanosarcina/proteins/protein_clustering/all_proteomes/kofam_scan/2026-04-29_kofam-scan_annotations_top-hits_filtered.tsv", # Columns: protein_id, KO 
    kegg_db_file="2026-03-10_KEGG-db_ko00001.csv"
)